### Import required libraries

In [54]:
# Snowpark for Python
#from snowflake import connector
#from snowflake.ml.utils import connection_params
from snowflake.snowpark import Session
from snowflake.snowpark.version import VERSION
from snowflake.snowpark.types import StructType, StructField, FloatType, StringType, IntegerType, List, PandasSeriesType
import snowflake.snowpark.functions as Fn
import snowflake.ml
from snowflake.ml.fileset import fileset

# data science libs
import numpy as np
import pandas as pd

# misc
import json

# torch
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import  DataLoader, Dataset, ConcatDataset

### Establish snowpark session

In [55]:
# Create Snowflake Session object
connection_parameters = json.load(open('cred.json'))
session = Session.builder.configs(connection_parameters).create()
session.sql_simplifier_enabled = True

snowflake_environment = session.sql('SELECT current_user(), current_version()').collect()
snowpark_version = VERSION

# Current Environment Details
print('User                        : {}'.format(snowflake_environment[0][0]))
print('Role                        : {}'.format(session.get_current_role()))
print('Database                    : {}'.format(session.get_current_database()))
print('Schema                      : {}'.format(session.get_current_schema()))
print('Warehouse                   : {}'.format(session.get_current_warehouse()))
print('Snowflake version           : {}'.format(snowflake_environment[0][1]))
print('Snowpark for Python version : {}.{}.{}'.format(snowpark_version[0],snowpark_version[1],snowpark_version[2]))

User                        : KNADADUR
Role                        : "PARTNER_APPS_USER_ROLE"
Database                    : "NEMO_DB"
Schema                      : "PUBLIC"
Warehouse                   : "COMPUTE_WH"
Snowflake version           : 7.34.0
Snowpark for Python version : 1.8.0


### Deploy registered model from snowflake for Inference

In [56]:
from snowflake.ml.registry import model_registry
from snowflake.ml._internal.utils import identifier

db = identifier._get_unescaped_name(session.get_current_database())
schema = identifier._get_unescaped_name(session.get_current_schema())

# will be a no-op if registry already exists
#model_registry.create_model_registry(session=session, database_name=db, schema_name=schema) 
registry = model_registry.ModelRegistry(session=session, database_name=db, schema_name=schema)
registry

model_name = "DICOM_pytorch_model_multigpu"
model_version = "v10"

model_ref = model_registry.ModelReference(registry=registry, 
                                          model_name=model_name,
                                          model_version=model_version)

model_ref.deploy(deployment_name="DICOM_pytorch_model_multigpu_v3", 
                 target_method="forward", 
                 permanent=True,
                 options={"relax":True})

Generated UDF file is persisted at: /tmp/tmphd727ec5.py


The version of package 'anyio' in the local environment is 3.7.0, which does not fit the criteria for the requirement 'anyio'. Your UDF might not work when the package version is different between the server and your local environment.
Package 'pytorch' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.
The version of package 'snowflake-snowpark-python' in the local environment is 1.8.0, which does not fit the criteria for the requirement 'snowflake-snowpark-python'. Your UDF might not work when the package version is different between the server and your local environment.


NEMO_DB.PUBLIC.DICOM_pytorch_model_multigpu_v3 is deployed to warehouse.


### Prepare test data for Inference

In [60]:
infer_folder = 'chest_xray/chest_xray/test/'

In [61]:
@torch.no_grad()
def test_loop(model,testdata, loss_fn, t_gpu):
    print('*'*5+'Testing Started'+'*'*5)
    # model.train(False)
    # model.eval()
    
    full_pred, full_lab = [], []
    
    TestLoss, TestAcc = 0.0, 0.0
    for data, target in testdata:
        # if t_gpu:
        #     data, target = data.cuda(), target.cuda()
        output = model_ref.predict(deployment_name=model, 
                          data=[data]) # can pass list of tensors (this will get converted to a pandas dataframe in the backend)

        df = pd.DataFrame([x[0][0] for x in output.values.tolist()])
        df["1"] = [x[0][1] for x in output.values.tolist()]

        # Convert pandas dataframe to tensors
        output = torch.tensor(df.values)
        loss = loss_fn(output, target)

        _, pred = torch.max(output.data, 1)
        TestLoss += loss.item() * data.size(0)
        TestAcc += torch.sum(pred == target.data)
        torch.cuda.empty_cache()
        full_pred += pred.tolist()
        full_lab += target.data.tolist()

    TestLoss = TestLoss / len(testdata.dataset)
    TestAcc = TestAcc / len(testdata.dataset)
    print(f'Loss: {TestLoss} Accuracy: {TestAcc}%')
    return full_pred, full_lab

### Define the test data and call the module

In [62]:
inferset = datasets.ImageFolder(infer_folder, 
                           transform=transforms.Compose([transforms.Resize(255),
                                                 transforms.CenterCrop(224),                                                              
                                                 transforms.ToTensor(),
                                                ]))
test_dl = DataLoader(inferset, batch_size=32)

### Call the Model for Inference

In [63]:
loss_fn = nn.CrossEntropyLoss()
model = "DICOM_pytorch_model_multigpu_v3"
pred, lab = test_loop(model, test_dl, loss_fn, False)

*****Testing Started*****
Loss: 0.37859742807989566 Accuracy: 0.9070512652397156%


### Compare Predicted and Original Labels

In [64]:
df_pred = pd.DataFrame(pred,columns=['pred'])
df_pred['label'] = lab
df_pred

,pred,label
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
...,...,...
619,1,1
620,1,1
621,1,1
622,1,1


### Extract Wrongly Predicted Labels

In [65]:
len(df_pred[df_pred['pred']!=df_pred['label']])

58

### Extract Correctly Predicted Labels

In [66]:
len(df_pred[df_pred['pred']==df_pred['label']])

566

In [67]:
session.close()